## Import check: `cfnresponse.send()` with a mocked HTTP client

In [ ]:
import json
from unittest import mock

import cfnresponse

print("Import check: cfnresponse.send() with a mocked HTTP client")
assert (cfnresponse.SUCCESS, cfnresponse.FAILED) == ("SUCCESS", "FAILED")

event = {
    "ResponseURL": "https://example.invalid/response",
    "StackId": "arn:aws:cloudformation:us-east-1:123456789012:stack/example-stack/1",
    "RequestId": "request-1",
    "LogicalResourceId": "ExampleResource",
}
context = mock.Mock(log_stream_name="example-log-stream")

# send() PUTs a JSON body to the event's ResponseURL. Patching the module's urllib3 pool keeps the
# request on this machine and lets the cell inspect exactly what would have been sent.
with mock.patch.object(cfnresponse.http, "request", return_value=mock.Mock(status=200)) as request:
    cfnresponse.send(event, context, cfnresponse.SUCCESS, {"key": "value"}, "example-physical-id")
method, url = request.call_args.args[:2]
body = json.loads(request.call_args.kwargs["body"])
print(f"✓ {method} {url}: Status={body['Status']}, PhysicalResourceId={body['PhysicalResourceId']}, Data={body['Data']}")
assert (method, url) == ("PUT", event["ResponseURL"]), (method, url)
assert (body["Status"], body["PhysicalResourceId"], body["Data"]) == ("SUCCESS", "example-physical-id", {"key": "value"}), body
assert [body[k] for k in ("StackId", "RequestId", "LogicalResourceId")] == [event[k] for k in ("StackId", "RequestId", "LogicalResourceId")], body

# Without a physical resource id or a reason, send() falls back to the Lambda log stream name.
with mock.patch.object(cfnresponse.http, "request", return_value=mock.Mock(status=200)) as request:
    cfnresponse.send(event, context, cfnresponse.FAILED, {})
body = json.loads(request.call_args.kwargs["body"])
assert body["Status"] == "FAILED" and body["PhysicalResourceId"] == "example-log-stream", body
assert "example-log-stream" in body["Reason"], body["Reason"]
print("✓ FAILED response: PhysicalResourceId and Reason default to the log stream name")

## Example 1: Basic cfnresponse Module Structure

In [ ]:
print("Example 1: Basic cfnresponse Module Structure")

try:
    # Note: cfnresponse is typically available in AWS Lambda Python runtime
    # For demonstration, we'll show the structure and concepts
    
    # Mock cfnresponse module structure (as it appears in AWS Lambda)
    print("✓ cfnresponse module structure:")
    
    # Main constants used in cfnresponse
    response_status = {
        'SUCCESS': 'SUCCESS',
        'FAILED': 'FAILED'
    }
    
    print("✓ Response status constants:")
    for status, value in response_status.items():
        print(f"  {status}: {value}")
    
    # Show cfnresponse main function signature
    print("✓ Main cfnresponse function:")
    print("  send(event, context, response_status, response_data, physical_resource_id, no_echo=False, reason=None)")
    
    # Parameters explanation
    parameters = {
        'event': 'CloudFormation event object from Lambda',
        'context': 'Lambda context object', 
        'response_status': 'SUCCESS or FAILED',
        'response_data': 'Dictionary of response data',
        'physical_resource_id': 'Unique ID for the custom resource',
        'no_echo': 'Boolean to mask response data in logs',
        'reason': 'Optional reason for failure'
    }
    
    print("✓ Function parameters:")
    for param, description in parameters.items():
        print(f"  {param}: {description}")
    
    # Show usage pattern
    print("✓ Typical usage pattern:")
    print("  1. Lambda receives CloudFormation event")
    print("  2. Process custom resource request")
    print("  3. Send response back to CloudFormation")
    print("  4. CloudFormation continues stack operation")
    
    # Show event types
    event_types = [
        'Create - Resource creation request',
        'Update - Resource update request', 
        'Delete - Resource deletion request'
    ]
    
    print("✓ CloudFormation custom resource event types:")
    for event_type in event_types:
        print(f"  - {event_type}")
    
    # Show response data structure
    print("✓ Response data structure:")
    response_example = {
        'Status': 'SUCCESS',
        'Reason': 'Resource created successfully',
        'PhysicalResourceId': 'custom-resource-123',
        'StackId': 'arn:aws:cloudformation:...',
        'RequestId': 'unique-request-id',
        'LogicalResourceId': 'MyCustomResource',
        'Data': {'key': 'value'}
    }
    
    for key, value in response_example.items():
        display_value = value if len(str(value)) < 50 else f"{str(value)[:50]}..."
        print(f"  {key}: {display_value}")
    
    print("✓ Basic cfnresponse module structure completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Note: cfnresponse is available in AWS Lambda Python runtime environment")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfnresponse example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Mock Custom Resource Lambda Handler

In [ ]:
import json
from unittest.mock import Mock

print("Example 2: Mock Custom Resource Lambda Handler")

try:
    # Mock Lambda handler for CloudFormation custom resource
    def mock_lambda_handler(event, context):
        """
        Mock Lambda handler that demonstrates cfnresponse usage pattern
        """
        print("✓ Lambda handler processing CloudFormation custom resource")
        
        # Extract event information
        request_type = event.get('RequestType', 'Unknown')
        resource_type = event.get('ResourceType', 'Unknown')
        logical_resource_id = event.get('LogicalResourceId', 'Unknown')
        
        print(f"  Request Type: {request_type}")
        print(f"  Resource Type: {resource_type}")
        print(f"  Logical Resource ID: {logical_resource_id}")
        
        # Mock response based on request type
        response_data = {}
        physical_resource_id = f"custom-resource-{logical_resource_id}-123"
        
        if request_type == 'Create':
            print("  Processing CREATE request")
            # Simulate resource creation
            response_data = {
                'ResourceId': physical_resource_id,
                'Status': 'Created',
                'Timestamp': '2024-01-01T10:00:00Z'
            }
            status = 'SUCCESS'
            reason = 'Resource created successfully'
            
        elif request_type == 'Update':
            print("  Processing UPDATE request")
            # Simulate resource update
            response_data = {
                'ResourceId': physical_resource_id,
                'Status': 'Updated', 
                'Timestamp': '2024-01-01T10:30:00Z'
            }
            status = 'SUCCESS'
            reason = 'Resource updated successfully'
            
        elif request_type == 'Delete':
            print("  Processing DELETE request")
            # Simulate resource deletion
            response_data = {
                'ResourceId': physical_resource_id,
                'Status': 'Deleted',
                'Timestamp': '2024-01-01T11:00:00Z'
            }
            status = 'SUCCESS'
            reason = 'Resource deleted successfully'
            
        else:
            print(f"  Unknown request type: {request_type}")
            status = 'FAILED'
            reason = f'Unknown request type: {request_type}'
        
        # This is where cfnresponse.send() would be called in real Lambda
        print("✓ Would call cfnresponse.send() with:")
        print(f"  Status: {status}")
        print(f"  Reason: {reason}")
        print(f"  Physical Resource ID: {physical_resource_id}")
        print(f"  Response Data: {response_data}")
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'status': status,
                'physicalResourceId': physical_resource_id,
                'data': response_data
            })
        }
    
    # Test with mock events
    test_events = [
        {
            'RequestType': 'Create',
            'ResourceType': 'Custom::TestResource',
            'LogicalResourceId': 'MyTestResource',
            'RequestId': 'test-request-123',
            'StackId': 'arn:aws:cloudformation:us-east-1:123456789012:stack/test-stack/12345'
        },
        {
            'RequestType': 'Update', 
            'ResourceType': 'Custom::TestResource',
            'LogicalResourceId': 'MyTestResource',
            'RequestId': 'test-request-456'
        },
        {
            'RequestType': 'Delete',
            'ResourceType': 'Custom::TestResource', 
            'LogicalResourceId': 'MyTestResource',
            'RequestId': 'test-request-789'
        }
    ]
    
    # Mock context object
    mock_context = Mock()
    mock_context.log_group_name = '/aws/lambda/test-function'
    mock_context.log_stream_name = '2024/01/01/test-stream'
    
    print("✓ Testing mock Lambda handler with different event types:")
    
    for i, event in enumerate(test_events, 1):
        print(f"\n--- Test Event {i} ---")
        result = mock_lambda_handler(event, mock_context)
        print(f"Handler result: {result['statusCode']}")
    
    print("✓ Mock custom resource Lambda handler completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfnresponse example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Response Data Structure and Validation

In [ ]:
import json
import urllib.parse

print("Example 3: Response Data Structure and Validation")

try:
    # Show CloudFormation custom resource response structure
    print("✓ CloudFormation custom resource response structure:")
    
    # Complete response structure
    complete_response = {
        "Status": "SUCCESS",  # or "FAILED"
        "Reason": "Resource operation completed successfully",
        "PhysicalResourceId": "custom-resource-unique-id",
        "StackId": "arn:aws:cloudformation:us-east-1:123456789012:stack/test-stack/12345",
        "RequestId": "unique-request-id-12345",
        "LogicalResourceId": "MyCustomResource",
        "NoEcho": False,
        "Data": {
            "CustomProperty1": "value1",
            "CustomProperty2": "value2",
            "OutputValue": "computed-result"
        }
    }
    
    print("✓ Complete response structure:")
    for key, value in complete_response.items():
        if isinstance(value, dict):
            print(f"  {key}: (dictionary with {len(value)} keys)")
            for sub_key, sub_value in value.items():
                print(f"    {sub_key}: {sub_value}")
        else:
            print(f"  {key}: {value}")
    
    # Validate required fields
    print("✓ Required response fields validation:")
    required_fields = ['Status', 'PhysicalResourceId', 'StackId', 'RequestId', 'LogicalResourceId']
    
    def validate_response(response):
        """Validate CloudFormation response structure"""
        missing_fields = []
        for field in required_fields:
            if field not in response:
                missing_fields.append(field)
        
        return len(missing_fields) == 0, missing_fields
    
    # Test validation
    is_valid, missing = validate_response(complete_response)
    print(f"  Response valid: {is_valid}")
    if missing:
        print(f"  Missing fields: {missing}")
    
    # Test with incomplete response
    incomplete_response = {
        "Status": "SUCCESS",
        "PhysicalResourceId": "test-resource"
        # Missing required fields
    }
    
    is_valid, missing = validate_response(incomplete_response)
    print(f"  Incomplete response valid: {is_valid}")
    if missing:
        print(f"  Missing fields: {missing}")
    
    # Show status values
    print("✓ Valid status values:")
    status_values = {
        "SUCCESS": "Operation completed successfully",
        "FAILED": "Operation failed - will cause stack rollback"
    }
    
    for status, description in status_values.items():
        print(f"  {status}: {description}")
    
    # Show data field usage
    print("✓ Data field usage patterns:")
    data_patterns = [
        "Return computed values for other resources to reference",
        "Provide status information about the custom operation",
        "Return configuration details or connection strings", 
        "Include metadata for debugging and monitoring"
    ]
    
    for pattern in data_patterns:
        print(f"  - {pattern}")
    
    # Show NoEcho usage
    print("✓ NoEcho field usage:")
    noecho_info = {
        "Purpose": "Hide sensitive data from CloudFormation logs",
        "Default": False,
        "Use cases": ["Passwords", "API keys", "Connection strings", "Sensitive configuration"]
    }
    
    for key, value in noecho_info.items():
        if isinstance(value, list):
            print(f"  {key}: {', '.join(value)}")
        else:
            print(f"  {key}: {value}")
    
    # Show physical resource ID best practices
    print("✓ Physical Resource ID best practices:")
    resource_id_practices = [
        "Use unique, immutable identifiers",
        "Include timestamp or UUID for uniqueness",
        "Keep consistent format across operations",
        "Make it meaningful for debugging",
        "Avoid sensitive information"
    ]
    
    for practice in resource_id_practices:
        print(f"  - {practice}")
    
    print("✓ Response data structure and validation completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfnresponse example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Error Handling and Failure Scenarios

In [ ]:
import json
import traceback

print("Example 4: Error Handling and Failure Scenarios")

try:
    # Mock error handling in Lambda custom resource
    print("✓ Error handling patterns for CloudFormation custom resources:")
    
    def mock_custom_resource_with_error_handling(event, context):
        """
        Demonstrates proper error handling in custom resource Lambda
        """
        response_data = {}
        physical_resource_id = event.get('PhysicalResourceId', 'default-resource-id')
        
        try:
            print("✓ Processing custom resource request")
            
            # Simulate different error scenarios
            request_type = event.get('RequestType')
            resource_properties = event.get('ResourceProperties', {})
            
            if request_type == 'Create':
                # Simulate creation logic that might fail
                required_property = resource_properties.get('RequiredProperty')
                if not required_property:
                    raise ValueError("RequiredProperty is missing")
                
                # Simulate successful creation
                physical_resource_id = f"resource-{required_property}-created"
                response_data = {
                    'ResourceId': physical_resource_id,
                    'CreatedAt': '2024-01-01T10:00:00Z',
                    'Status': 'Active'
                }
                
                print(f"  Resource created successfully: {physical_resource_id}")
                return send_response(event, 'SUCCESS', response_data, physical_resource_id)
                
            elif request_type == 'Update':
                # Simulate update logic
                print("  Processing update request")
                response_data = {'UpdatedAt': '2024-01-01T11:00:00Z'}
                return send_response(event, 'SUCCESS', response_data, physical_resource_id)
                
            elif request_type == 'Delete':
                # Simulate deletion - even if it fails, we should return SUCCESS
                print("  Processing delete request")
                try:
                    # Simulate deletion operation that might fail
                    if resource_properties.get('ForceFailDelete'):
                        raise Exception("Simulated deletion failure")
                    
                    response_data = {'DeletedAt': '2024-01-01T12:00:00Z'}
                    return send_response(event, 'SUCCESS', response_data, physical_resource_id)
                    
                except Exception as delete_error:
                    print(f"  Delete operation failed: {delete_error}")
                    # For delete operations, we often return SUCCESS even if deletion fails
                    # to prevent CloudFormation stack from getting stuck
                    response_data = {'DeleteError': str(delete_error)}
                    return send_response(event, 'SUCCESS', response_data, physical_resource_id, 
                                       reason="Delete operation failed but marked as success to prevent stack issues")
                
        except Exception as e:
            print(f"  Error in custom resource processing: {e}")
            error_response = {
                'Error': str(e),
                'ErrorType': type(e).__name__,
                'Traceback': traceback.format_exc()
            }
            return send_response(event, 'FAILED', error_response, physical_resource_id, reason=str(e))
    
    def send_response(event, status, response_data, physical_resource_id, reason=None):
        """Mock cfnresponse.send() function"""
        response = {
            'Status': status,
            'Reason': reason or f'Operation {status.lower()}',
            'PhysicalResourceId': physical_resource_id,
            'StackId': event.get('StackId', 'mock-stack'),
            'RequestId': event.get('RequestId', 'mock-request'),
            'LogicalResourceId': event.get('LogicalResourceId', 'mock-resource'),
            'Data': response_data
        }
        
        print(f"✓ Sending response: {status}")
        print(f"  Physical Resource ID: {physical_resource_id}")
        print(f"  Data keys: {list(response_data.keys())}")
        if reason:
            print(f"  Reason: {reason}")
        
        return response
    
    # Test different error scenarios
    test_scenarios = [
        {
            'name': 'Successful Create',
            'event': {
                'RequestType': 'Create',
                'ResourceProperties': {'RequiredProperty': 'test-value'},
                'LogicalResourceId': 'TestResource'
            }
        },
        {
            'name': 'Failed Create - Missing Property',
            'event': {
                'RequestType': 'Create',
                'ResourceProperties': {},  # Missing required property
                'LogicalResourceId': 'TestResource'
            }
        },
        {
            'name': 'Successful Update',
            'event': {
                'RequestType': 'Update',
                'PhysicalResourceId': 'existing-resource-123',
                'LogicalResourceId': 'TestResource'
            }
        },
        {
            'name': 'Delete with Failure',
            'event': {
                'RequestType': 'Delete',
                'PhysicalResourceId': 'existing-resource-123',
                'ResourceProperties': {'ForceFailDelete': True},
                'LogicalResourceId': 'TestResource'
            }
        }
    ]
    
    print("✓ Testing error handling scenarios:")
    
    for scenario in test_scenarios:
        print(f"\n--- {scenario['name']} ---")
        mock_context = {'log_stream_name': 'test-stream'}
        result = mock_custom_resource_with_error_handling(scenario['event'], mock_context)
        print(f"Final status: {result['Status']}")
    
    # Show best practices for error handling
    print("\n✓ Error handling best practices:")
    best_practices = [
        "Always send a response to prevent CloudFormation timeout",
        "Include detailed error information in response data",
        "Use appropriate status (SUCCESS/FAILED) based on operation outcome",
        "For Delete operations, consider returning SUCCESS even on failure",
        "Log errors for debugging but don't expose sensitive information",
        "Set meaningful physical resource IDs for tracking",
        "Include proper error reasons for CloudFormation logs"
    ]
    
    for practice in best_practices:
        print(f"  - {practice}")
    
    print("✓ Error handling and failure scenarios completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfnresponse example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: CloudFormation Template Integration

In [ ]:
import json

print("Example 5: CloudFormation Template Integration")

try:
    # Show how cfnresponse integrates with CloudFormation templates
    print("✓ CloudFormation template integration with custom resources:")
    
    # Example CloudFormation template with custom resource
    cf_template_with_custom_resource = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "Template with custom resource using cfnresponse",
        "Resources": {
            "CustomResourceLambdaRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "AssumeRolePolicyDocument": {
                        "Version": "2012-10-17",
                        "Statement": [{
                            "Effect": "Allow",
                            "Principal": {"Service": "lambda.amazonaws.com"},
                            "Action": "sts:AssumeRole"
                        }]
                    },
                    "ManagedPolicyArns": [
                        "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
                    ]
                }
            },
            "CustomResourceLambda": {
                "Type": "AWS::Lambda::Function",
                "Properties": {
                    "FunctionName": "CustomResourceHandler",
                    "Runtime": "python3.9",
                    "Handler": "index.lambda_handler",
                    "Role": {"Fn::GetAtt": ["CustomResourceLambdaRole", "Arn"]},
                    "Code": {
                        "ZipFile": """
import cfnresponse
import json

def lambda_handler(event, context):
    try:
        request_type = event['RequestType']
        properties = event.get('ResourceProperties', {})
        
        if request_type == 'Create':
            # Custom logic here
            response_data = {'Message': 'Resource created'}
            cfnresponse.send(event, context, cfnresponse.SUCCESS, response_data)
            
        elif request_type == 'Update':
            # Custom update logic
            response_data = {'Message': 'Resource updated'}
            cfnresponse.send(event, context, cfnresponse.SUCCESS, response_data)
            
        elif request_type == 'Delete':
            # Custom delete logic
            response_data = {'Message': 'Resource deleted'}
            cfnresponse.send(event, context, cfnresponse.SUCCESS, response_data)
            
    except Exception as e:
        cfnresponse.send(event, context, cfnresponse.FAILED, {}, reason=str(e))
"""
                    },
                    "Timeout": 60
                }
            },
            "MyCustomResource": {
                "Type": "Custom::MyCustomResource",
                "Properties": {
                    "ServiceToken": {"Fn::GetAtt": ["CustomResourceLambda", "Arn"]},
                    "CustomProperty1": "value1",
                    "CustomProperty2": "value2"
                }
            }
        },
        "Outputs": {
            "CustomResourceResponse": {
                "Description": "Response from custom resource",
                "Value": {"Fn::GetAtt": ["MyCustomResource", "Message"]}
            }
        }
    }
    
    print("✓ CloudFormation template structure:")
    print(f"  Template has {len(cf_template_with_custom_resource['Resources'])} resources")
    
    # Show custom resource definition
    custom_resource = cf_template_with_custom_resource["Resources"]["MyCustomResource"]
    print("✓ Custom resource definition:")
    print(f"  Type: {custom_resource['Type']}")
    print(f"  ServiceToken: Lambda function ARN reference")
    print(f"  Properties: {len(custom_resource['Properties'])} custom properties")
    
    # Show Lambda function integration
    lambda_function = cf_template_with_custom_resource["Resources"]["CustomResourceLambda"]
    print("✓ Lambda function for custom resource:")
    print(f"  Runtime: {lambda_function['Properties']['Runtime']}")
    print(f"  Handler: {lambda_function['Properties']['Handler']}")
    print("  Contains cfnresponse.send() calls")
    
    # Show workflow
    print("✓ Custom resource workflow:")
    workflow_steps = [
        "1. CloudFormation creates/updates/deletes stack",
        "2. CloudFormation invokes Lambda function via ServiceToken",
        "3. Lambda receives event with RequestType and ResourceProperties",
        "4. Lambda processes custom logic based on RequestType",
        "5. Lambda calls cfnresponse.send() with status and data",
        "6. CloudFormation receives response and continues operation",
        "7. Custom resource data available via Fn::GetAtt"
    ]
    
    for step in workflow_steps:
        print(f"  {step}")
    
    # Show best practices for template integration
    print("✓ Template integration best practices:")
    integration_practices = [
        "Use descriptive custom resource types (Custom::MyResourceType)",
        "Include proper IAM permissions for Lambda function",
        "Set appropriate timeout values for Lambda function",
        "Use Fn::GetAtt to reference custom resource outputs",
        "Include proper error handling in Lambda code",
        "Use DependsOn when custom resources depend on other resources",
        "Version Lambda functions for updates"
    ]
    
    for practice in integration_practices:
        print(f"  - {practice}")
    
    # Show common use cases
    print("✓ Common custom resource use cases:")
    use_cases = [
        "Initialize databases or external services",
        "Generate random passwords or certificates",
        "Call external APIs during stack operations",
        "Custom validation or compliance checks",
        "Clean up resources not managed by CloudFormation",
        "Bridge CloudFormation with third-party services"
    ]
    
    for use_case in use_cases:
        print(f"  - {use_case}")
    
    # Show output usage
    print("✓ Using custom resource outputs:")
    print("  - Fn::GetAtt: ['MyCustomResource', 'PropertyName']")
    print("  - Ref: MyCustomResource (returns PhysicalResourceId)")
    print("  - Export values for cross-stack references")
    print("  - Condition-based resource creation")
    
    print("✓ CloudFormation template integration completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfnresponse example (cell 11) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")